# Supervised Classification & Model Tuning — Customer Churn

**Milestone 04:** Train at least four classification models, tune hyperparameters with stratified cross-validation, evaluate with Precision, Recall, F1, ROC-AUC and a confusion matrix, and save the selected model.

**This version is fixed for the columns shown in your CSV:** `customer_id`, `tenure_months`, `support_tickets`, `monthly_spend_inr`, `last_login_days`, `plan_type`, `churned`.

**Important:** Upload `customer-churn-training.csv` into the Colab Files panel before running. Then use **Runtime → Run all** or run cells from top to bottom.

In [ ]:
# 1. Imports and settings
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

RANDOM_STATE = 42
DATA_PATH = "/content/customer-churn-training.csv"
TARGET_COLUMN = "churned"
ID_COLUMNS = ["customer_id"]
print("Settings ready. Target column:", TARGET_COLUMN)


In [ ]:
# 2. Load and inspect dataset
if not os.path.exists(DATA_PATH):
    # Fallback if the notebook is running in the current working directory
    DATA_PATH = "customer-churn-training.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "CSV file not found. In Colab, click the folder icon on the left, "
        "upload customer-churn-training.csv, then run this cell again."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower()

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
display(df.dtypes.to_frame("dtype"))
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        f"Available columns: {df.columns.tolist()}"
    )

print("Target distribution:")
display(df[TARGET_COLUMN].value_counts(dropna=False).to_frame("count"))


## 3. Clean data and define target

The target column is `churned` (lowercase). Identifier columns such as `customer_id` are excluded from model features to avoid using an ID as a predictor. The notebook supports binary labels such as Yes/No or 0/1.

In [ ]:
# Remove rows with missing target and duplicate records
df = df.dropna(subset=[TARGET_COLUMN]).drop_duplicates().copy()

id_cols = [c for c in ID_COLUMNS if c in df.columns]
X = df.drop(columns=[TARGET_COLUMN] + id_cols).copy()
y_raw = df[TARGET_COLUMN].copy()

if y_raw.nunique() != 2:
    raise ValueError(
        f"Expected a binary target with exactly 2 classes, but found "
        f"{y_raw.nunique()} classes: {y_raw.unique().tolist()}"
    )

# Encode target to 0/1
if pd.api.types.is_numeric_dtype(y_raw):
    classes = sorted(y_raw.unique().tolist())
    y = y_raw.map({classes[0]: 0, classes[1]: 1}).astype(int)
else:
    y_text = y_raw.astype(str).str.strip().str.lower()
    common_map = {
        "yes": 1, "true": 1, "1": 1, "churn": 1, "churned": 1,
        "no": 0, "false": 0, "0": 0, "stay": 0, "not churn": 0,
        "not churned": 0
    }
    if set(y_text.unique()).issubset(set(common_map)):
        y = y_text.map(common_map).astype(int)
    else:
        classes = sorted(y_text.unique().tolist())
        y = y_text.map({classes[0]: 0, classes[1]: 1}).astype(int)
        print("Target encoding used:", {classes[0]: 0, classes[1]: 1})

print("Rows after cleaning:", len(X))
print("Features used:", X.columns.tolist())
print("Target balance (proportion):")
display(y.value_counts(normalize=True).rename("proportion").to_frame())


In [ ]:
# 4. Train/test split and preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 5. Train and tune four models

Grid search uses stratified 5-fold cross-validation on the training split and ROC-AUC as the selection metric. Preprocessing is inside each pipeline to reduce data leakage.

In [ ]:
models_and_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        {"model__C": [0.1, 1.0, 10.0]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {"model__max_depth": [None, 5, 10], "model__min_samples_leaf": [1, 5]}
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        {"model__n_estimators": [100, 200], "model__max_depth": [None, 10], "model__min_samples_leaf": [1, 3]}
    ),
    "Extra Trees": (
        ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        {"model__n_estimators": [100, 200], "model__max_depth": [None, 10], "model__min_samples_leaf": [1, 3]}
    )
}

# If either class has fewer than 5 training examples, reduce folds safely.
min_class_count = int(y_train.value_counts().min())
n_splits = min(5, min_class_count)
if n_splits < 2:
    raise ValueError("Not enough examples per target class for cross-validation.")
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

searches = {}
cv_rows = []
for name, (estimator, grid) in models_and_grids.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    search = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        error_score="raise"
    )
    search.fit(X_train, y_train)
    searches[name] = search
    cv_rows.append({
        "Model": name,
        "Best CV ROC-AUC": search.best_score_,
        "Best parameters": str(search.best_params_)
    })
    print(f"{name}: best CV ROC-AUC = {search.best_score_:.4f}")

cv_results = pd.DataFrame(cv_rows).sort_values("Best CV ROC-AUC", ascending=False)
display(cv_results)


In [ ]:
# 6. Evaluate models on the held-out test set
test_rows = []
for name, search in searches.items():
    fitted_model = search.best_estimator_
    predictions = fitted_model.predict(X_test)
    probabilities = fitted_model.predict_proba(X_test)[:, 1]
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-Score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

test_results = pd.DataFrame(test_rows).sort_values("ROC-AUC", ascending=False)
display(test_results.round(4))


In [ ]:
# 7. Plot ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, search in searches.items():
    RocCurveDisplay.from_estimator(search.best_estimator_, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_title("ROC Curves — Test Set")
plt.tight_layout()
plt.show()


In [ ]:
# 8. Confusion matrix and detailed report for selected model
champion_name = test_results.iloc[0]["Model"]
champion_model = searches[champion_name].best_estimator_
champion_predictions = champion_model.predict(X_test)

print("Selected model (highest test ROC-AUC for this comparison):", champion_name)
print(classification_report(y_test, champion_predictions, zero_division=0))

ConfusionMatrixDisplay.from_estimator(champion_model, X_test, y_test)
plt.title(f"Confusion Matrix — {champion_name}")
plt.tight_layout()
plt.show()


In [ ]:
# 9. Save model and evidence files
joblib.dump(champion_model, "champion_churn_model.joblib")
test_results.to_csv("model_comparison_results.csv", index=False)
cv_results.to_csv("cross_validation_results.csv", index=False)
pd.Series(X.columns, name="feature").to_csv("model_feature_schema.csv", index=False)

print("Saved files:")
print("- champion_churn_model.joblib")
print("- model_comparison_results.csv")
print("- cross_validation_results.csv")
print("- model_feature_schema.csv")


## 10. Responsible-use notes

- Confirm the dataset source, permission/license, and intended population before sharing.
- Exclude direct identifiers and any fields recorded only after churn occurred.
- Metrics describe this dataset and split; they do not guarantee performance on future customers.
- False positives and false negatives have different business costs; choose any deployment threshold with documented review.
- Use predictions to guide supportive outreach, not to deny services or penalize customers.
- Monitor performance over time and across relevant groups where lawful and appropriate.